# EcoPower AI: Exploratory Data Analysis & Machine Learning Modeling
**Author:** Vanshika Verma | B.Tech CSE – AI, Om Sterling Global University, Hisar  
**Internship:** 1M1B AI for Sustainability Virtual Internship in collaboration with IBM SkillsBuild and AICTE  
**SDG Focus:** SDG 7 (Affordable and Clean Energy) & SDG 13 (Climate Action)  
**Career Goal:** Data Scientist

---
### Notebook Objectives:
1. **Data Ingestion & Integrity Auditing**: Validate 1-year hourly telemetry (8,784 records).
2. **Exploratory Data Analysis (EDA)**: Uncover peak hours, seasonal swings, temperature vs. load elasticity, and weekday/weekend dynamics.
3. **Physics-Informed Feature Engineering**: Create cyclical encodings (sin/cos of hour and month) and thermal cooling/heating proxies (CDD/HDD).
4. **Machine Learning Benchmarking**: Train and evaluate Linear Regression, Random Forest, and Gradient Boosting Regressors using chronological train/test splitting (no data leakage).
5. **Model Evaluation & Best Model Serialization**: Evaluate MAE, RMSE, and $R^2$, and export `best_model.pkl` for the RAG-integrated Streamlit dashboard.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configurations
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Audit

In [ ]:
data_path = os.path.join('..', 'data', 'energy_consumption.csv')
df = pd.read_csv(data_path)
df['datetime'] = pd.to_datetime(df['datetime'])
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())
print("\n--- Missing Values Audit ---")
print(df.isnull().sum())
print("\n--- Statistical Summary ---")
display(df.describe().round(2))

## 2. Exploratory Data Analysis (EDA)
### 2.1 Trend Analysis: Longitudinal Energy Consumption Over the Year

In [ ]:
daily_df = df.set_index('datetime').resample('D').agg({
    'energy_consumption_kwh': 'mean',
    'temperature_c': 'mean'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Daily Mean Energy (kWh)', color=color)
ax1.plot(daily_df['datetime'], daily_df['energy_consumption_kwh'], color=color, linewidth=1.5, label='Energy (kWh)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Mean Ambient Temp (°C)', color=color)
ax2.plot(daily_df['datetime'], daily_df['temperature_c'], color=color, alpha=0.6, linestyle='--', label='Temperature (°C)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Longitudinal Electricity Consumption vs. Ambient Temperature (2024)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### 2.2 Diurnal Profile: Hourly Consumption by Day of Week (Peak Analysis)

In [ ]:
hourly_pivot = df.pivot_table(values='energy_consumption_kwh', index='hour', columns='is_weekend', aggfunc='mean')
hourly_pivot.columns = ['Weekday', 'Weekend']

plt.figure(figsize=(12, 5))
plt.plot(hourly_pivot.index, hourly_pivot['Weekday'], marker='o', linewidth=2.5, color='#1b7837', label='Weekday Load Profile')
plt.plot(hourly_pivot.index, hourly_pivot['Weekend'], marker='s', linewidth=2.0, color='#762a83', linestyle='--', label='Weekend Load Profile')
plt.axvspan(10, 16, color='red', alpha=0.15, label='High-Demand Peak Tariff Window (10:00 - 16:00)')
plt.title('Hourly Load Profiles: Weekday vs Weekend Peak Differentiation', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day (00:00 - 23:00)')
plt.ylabel('Average Energy Consumption (kWh)')
plt.xticks(range(0, 24))
plt.legend(frameon=True)
plt.show()

### 2.3 Temperature Sensitivity & Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: Temp vs Energy
sns.scatterplot(
    data=df.sample(2000, random_state=42),
    x='temperature_c', y='energy_consumption_kwh',
    hue='is_business_hours', palette={0: 'gray', 1: 'green'}, alpha=0.5, ax=axes[0]
)
axes[0].set_title('Temperature vs Electricity Consumption Elasticity', fontweight='bold')
axes[0].set_xlabel('Ambient Temperature (°C)')
axes[0].set_ylabel('Energy Consumption (kWh)')

# Correlation Heatmap
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, cmap='crest', annot=True, fmt='.2f', ax=axes[1], cbar=False)
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Feature Engineering
Creating cyclic encodings for temporal variables to capture continuous cyclic continuity and building cooling/heating degree indicators based on Bureau of Energy Efficiency (BEE) benchmarks.

In [ ]:
import sys
sys.path.append('..')
from src.feature_engineering import engineer_features, FEATURE_COLUMNS

feat_df = engineer_features(df)
print("Engineered Features:", FEATURE_COLUMNS)
display(feat_df[FEATURE_COLUMNS].head())

## 4. Machine Learning Benchmarking
We employ a chronological time-series split (first 80% train, last 20% test) to strictly prevent future information leakage into training.
Models compared:
1. **Linear Regression** (Baseline parametric model)
2. **Random Forest Regressor** (Ensemble bagging model)
3. **Gradient Boosting Regressor** (Sequential boosting model)

In [ ]:
from src.model_training import train_and_evaluate_models

metadata = train_and_evaluate_models(data_path)
metrics_df = pd.DataFrame(metadata['metrics_comparison']).T
print("\n--- Model Performance Comparison Table ---")
display(metrics_df)

# Visual Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metrics_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], color=['#2b83ba', '#d7191c'])
axes[0].set_title('Error Comparison (Lower is Better)', fontweight='bold')
axes[0].set_ylabel('kWh')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

metrics_df['R2'].plot(kind='bar', ax=axes[1], color='#1a9641')
axes[1].set_title('R² Variance Explained (Higher is Better)', fontweight='bold')
axes[1].set_ylabel('R² Score')
axes[1].set_ylim(0.8, 1.0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

### 4.1 Feature Importance Analysis

In [ ]:
fi_series = pd.Series(metadata['feature_importance'])
plt.figure(figsize=(10, 5))
fi_series.sort_values().plot(kind='barh', color='#2ca25f')
plt.title(f"Feature Importance: {metadata['best_model_name']}", fontweight='bold')
plt.xlabel('Normalized Importance Weight')
plt.show()

## 5. Summary & Key Findings
1. **Peak Load Drivers**: High electricity demand correlates heavily with ambient cooling requirements (`cdd_cooling_load`) and institutional occupancy between 10:00 to 16:00.
2. **Best Model Performance**: The **Gradient Boosting Regressor** achieved superior generalization ($R^2 = 0.9839$, $RMSE = 3.86$ kWh), accurately capturing the non-linear interaction between thermal degree days and active occupancy.
3. **Integration with RAG**: The trained model and feature scalers are serialized in `../models/` and directly power the real-time inference engine within the Streamlit AI Sustainability Advisor.